# Land Cover Prediction Maps
## GEOG 6160 Final Project
## Magnus Tveit

Loads `landcover_model.keras` (trained in `LandCoverCNN.ipynb`) and predicts 4-class
land cover for all 14 years of Landsat imagery using a sliding window approach.

**Prerequisites:** Run `LandCoverCNN.ipynb` first to produce `landcover_model.keras`.

Classes:
```
Lake          — #2166ac (dark blue)
River         — #74add1 (light blue)
Sediment Bank — #d9ef8b (yellow-green)
Rock          — #a05000 (brown)
```

### Set Up

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import rasterio
from tensorflow import keras

### Settings

In [ ]:
base_dir   = os.getcwd()
data_dir   = os.path.join(base_dir, "..", "Data") 
model_path = os.path.join(base_dir, "landcover_model.keras")

CHIP_SIZE     = 8        # must match training
NUM_CHANNELS  = 7
SCALE         = 0.0000275
OFFSET        = -0.2
STRIDE        = 4        # sliding window step — smaller = smoother map but slower
PIXEL_AREA_M2 = 30 * 30  # Landsat pixel footprint in square metres

# Must match folder names and order from LandCoverExtractChips exactly
CLASSES      = ["Lake", "River", "Sediment Bank", "Rock"]
NUM_CLASSES  = len(CLASSES)
CLASS_COLORS = ["#2166ac", "#74add1", "#d9ef8b", "#a05000"]   # one colour per class

BAND_SUFFIXES  = ["_SR_B2.TIF", "_SR_B3.TIF", "_SR_B4.TIF",
                  "_SR_B5.TIF", "_SR_B6.TIF", "_SR_NDVI.TIF", "_SR_NDWI.TIF"]
TRAINING_YEARS = [2013, 2016, 2020, 2022]

# Build colourmap once — reused in all classification plots
cmap = ListedColormap(CLASS_COLORS)
norm = BoundaryNorm(boundaries=[-0.5 + i for i in range(NUM_CLASSES + 1)],
                    ncolors=NUM_CLASSES)

model = keras.models.load_model(model_path)

### Helper functions

In [ ]:
def stack_bands(year_dir):
    """Load and scale 7 bands. Returns (stacked array, transform, profile)."""
    tif_files  = os.listdir(year_dir)
    band_paths = []
    for suffix in BAND_SUFFIXES:
        match = next((f for f in tif_files if f.endswith(suffix)), None)
        if match is None:
            raise FileNotFoundError(f"Missing {suffix} in {year_dir}")
        band_paths.append(os.path.join(year_dir, match))

    with rasterio.open(band_paths[0]) as src:
        transform = src.transform
        profile   = src.profile.copy()

    bands = []
    for i, bp in enumerate(band_paths):
        with rasterio.open(bp) as src:
            b = src.read(1).astype("float32")
        # Apply Landsat scaling to spectral bands; clip index bands to [-1, 1]
        b = np.clip(b*SCALE+OFFSET,0,1) if i<5 else np.clip(b,-1,1)
        bands.append(b)

    return np.stack(bands, axis=0), transform, profile


def predict_class_map(stacked):
    """Returns (H,W) integer class map and (H,W,NUM_CLASSES) probability cube."""
    n_bands, H, W = stacked.shape
    half = CHIP_SIZE // 2

    # Initialise probability cube with NaN — gaps are filled after prediction
    prob_map = np.full((H, W, NUM_CLASSES), np.nan, dtype="float32")
    chip_list, centre_list = [], []

    for r in range(half, H-half, STRIDE):
        for c in range(half, W-half, STRIDE):
            chip = stacked[:, r-half:r+half, c-half:c+half]
            chip_list.append(np.moveaxis(chip, 0, -1))
            centre_list.append((r, c))

    if not chip_list:
        return np.zeros((H,W), dtype="int32"), prob_map

    chips_array = np.stack(chip_list).astype("float32")
    preds = model.predict(chips_array, batch_size=64, verbose=0)

    # Write each chip's softmax output into the probability cube
    for (r, c), p in zip(centre_list, preds):
        prob_map[r, c, :] = p

    # Fill NaN gaps per class using neighbourhood mean
    from scipy.ndimage import generic_filter
    for k in range(NUM_CLASSES):
        layer = prob_map[:,:,k]
        if np.isnan(layer).any():
            def fill_nan(v):
                centre = v[len(v)//2]
                if not np.isnan(centre): return centre
                valid = v[~np.isnan(v)]
                return valid.mean() if len(valid) else 1.0/NUM_CLASSES
            prob_map[:,:,k] = generic_filter(layer, fill_nan,
                                             size=STRIDE*2+1, mode="nearest")

    # Take the class with the highest probability at each pixel
    class_map = np.argmax(prob_map, axis=2).astype("int32")
    return class_map, prob_map

### Run predictions for all years

In [ ]:
year_folders = sorted([f for f in os.listdir(data_dir)
                        if os.path.isdir(os.path.join(data_dir, f))])
results = {}

for year_folder in year_folders:
    year     = int(year_folder)
    year_dir = os.path.join(data_dir, year_folder)

    try:
        stacked, transform, profile = stack_bands(year_dir)
    except FileNotFoundError as e:
        print(f"  {year}: SKIPPED — {e}"); continue

    class_map, prob_map = predict_class_map(stacked)

    # Calculate area per class from pixel counts
    areas = {}
    for i, cls in enumerate(CLASSES):
        n_px = (class_map == i).sum()
        areas[cls] = (n_px * PIXEL_AREA_M2) / 1e6

    results[year] = dict(class_map=class_map, prob_map=prob_map,
                         areas=areas, stacked=stacked)

    area_str = "  ".join(f"{cls}={areas[cls]:.2f}km²" for cls in CLASSES)
    tag = " [TRAINING]" if year in TRAINING_YEARS else ""
    print(f"{year}: {area_str}{tag}")

print(f"\nDone — {len(results)} years.")

### Per-year maps

In [ ]:
for year, res in sorted(results.items()):
    stacked   = res["stacked"]
    class_map = res["class_map"]
    prob_map  = res["prob_map"]
    areas     = res["areas"]
    tag       = " [training year]" if year in TRAINING_YEARS else ""

    # Build NIR/Red/Green false colour composite
    rgb = stacked[[3, 2, 1], :, :].transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

    fig, axes = plt.subplots(3, 2, figsize=(10, 14))
    fig.suptitle(f"{year}{tag}", fontsize=14, fontweight="bold")

    # Top row: false colour (left) + classification map (right)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("False colour (NIR/Red/Green)", fontsize=10)
    axes[0, 0].axis("off")

    axes[0, 1].imshow(class_map, cmap=cmap, norm=norm, interpolation="nearest")
    axes[0, 1].set_title("Land cover classification", fontsize=10)
    axes[0, 1].axis("off")
    patches = [
        mpatches.Patch(color=CLASS_COLORS[i],
                       label=f"{CLASSES[i]} ({areas[CLASSES[i]]:.2f} km²)")
        for i in range(NUM_CLASSES)
    ]
    axes[0, 1].legend(handles=patches, loc="lower right", fontsize=7, framealpha=0.85)

    # Rows 2–3: probability maps, one per class
    prob_axes = [axes[1, 0], axes[1, 1], axes[2, 0], axes[2, 1]]
    for i, (ax, cls) in enumerate(zip(prob_axes, CLASSES)):
        im = ax.imshow(prob_map[:, :, i], cmap="RdYlGn", vmin=0, vmax=1)
        ax.set_title(f"P({cls})", fontsize=10)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

## 6. Summary grid
Training years are marked with ★.

In [ ]:
years = sorted(results.keys())
ncols = 5
nrows = (len(years) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
axes = axes.flatten()

for i, year in enumerate(years):
    class_map = results[year]["class_map"]
    axes[i].imshow(class_map, cmap=cmap, norm=norm, interpolation="nearest")
    tag = " ★" if year in TRAINING_YEARS else ""
    axes[i].set_title(f"{year}{tag}", fontsize=9)
    axes[i].axis("off")

# Hide unused subplot slots
for j in range(i+1, len(axes)):
    axes[j].axis("off")

patches = [mpatches.Patch(color=CLASS_COLORS[i], label=CLASSES[i])
           for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc="lower center", ncol=NUM_CLASSES,
           fontsize=10, bbox_to_anchor=(0.5, 0.01))
fig.suptitle("Land Cover Classification — Lake Powell (2013–2026)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

### Area time series
Gray shading marks training years.

In [ ]:
years_list = sorted(results.keys())

fig, ax = plt.subplots(figsize=(13, 5))

# Shade training years
for yr in TRAINING_YEARS:
    ax.axvspan(yr-0.4, yr+0.4, alpha=0.10, color="gray")

# Plot one line per class using its assigned colour
for i, cls in enumerate(CLASSES):
    areas_list = [results[y]["areas"][cls] for y in years_list]
    ax.plot(years_list, areas_list, "o-", color=CLASS_COLORS[i],
            linewidth=2, markersize=6, label=cls)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Area (km²)", fontsize=12)
ax.set_title("Land Cover Area — Lake Powell (2013–2026)",
             fontsize=13, fontweight="bold")
ax.set_xticks(years_list)
ax.set_xticklabels([str(y) for y in years_list], rotation=45)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()

print("\nArea summary (km²):")
header = f"{'Year':>6}  " + "  ".join(f"{cls:>13}" for cls in CLASSES)
print(header)
print("-" * len(header))
for yr in years_list:
    row = f"{yr:>6}  " + "  ".join(f"{results[yr]['areas'][cls]:>13.3f}" for cls in CLASSES)
    print(row)